Tutorial: https://python.langchain.com/docs/integrations/graphs/neo4j_cypher/

In [1]:
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
NEO_PASS = os.environ.get("NEO_PASS")
NEO_USER = os.environ.get("NEO_USER")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
NEO_DATABASE = os.environ.get("NEO_DATABASE")

In [3]:
graph = Neo4jGraph(url="bolt://localhost:7687", 
                   username="neo4j", 
                   password="finkg v2",
                   database="finkg")

In [4]:
graph

In [5]:
print(graph.schema)

Node properties:
Company {name: STRING}
Industry {name: STRING}
SP {name: STRING}
Exchange {name: STRING}
SecFilings {name: STRING}
State {name: STRING}
Person {name: STRING}
Major_industry_group {name: STRING}
Event {name: STRING}
Event_Type {name: STRING}
Relationship properties:

The relationships:
(:Company)-[:BELONGS_TO_INDUSTRY_OF]->(:Industry)
(:Company)-[:HAS_EXCHANGE_MARKET]->(:Exchange)
(:Company)-[:HAS_SEC_FILINGS]->(:SecFilings)
(:Company)-[:HAS_STATE_LOCATION]->(:State)
(:Company)-[:HAS_STATE_OF_INCORPORATION]->(:State)
(:Company)-[:IS_PARTIAL_OWNER_OF]->(:Company)
(:Company)-[:IS_PARTIAL_OWNER_OF]->(:SP)
(:Industry)-[:INDUSTRY_BELONGS_TO_MAJOR_GROUP]->(:Major_industry_group)
(:SP)-[:BELONGS_TO_INDUSTRY_OF]->(:Industry)
(:SP)-[:HAS_EXCHANGE_MARKET]->(:Exchange)
(:SP)-[:HAS_SEC_FILINGS]->(:SecFilings)
(:SP)-[:HAS_STATE_LOCATION]->(:State)
(:SP)-[:HAS_STATE_OF_INCORPORATION]->(:State)
(:SP)-[:IS_PARTIAL_OWNER_OF]->(:Company)
(:SP)-[:IS_PARTIAL_OWNER_OF]->(:SP)
(:SP)-[:IS_DIR

Lets start by quering the full graph and lets see how efficient the llmo is.

In [6]:
chain = GraphCypherQAChain.from_llm(
    llm=ChatOpenAI(temperature=0.5), 
    graph=graph, verbose=True, 
    allow_dangerous_requests=True,
    return_intermediate_steps=True,
)

In [ ]:
chain.invoke({"query": "What are the compan that impact company 0000022356?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (:Company {name: '0000022356'})-[:IMPACTS]->(e:Event)
RETURN e;
Full Context:
[]

> Finished chain.


{'query': 'What are the events that impact company 0000022356?',
 'result': "I don't know the answer.",
 'intermediate_steps': [{'query': "MATCH (:Company {name: '0000022356'})-[:IMPACTS]->(e:Event)\nRETURN e;"},
  {'context': []}]}

In [23]:
chain.invoke({"query": "About the company with CIK 0000318154, how have past events impacted the company and its sector? is the company exposed to specific political or regulatory risks that have historically impacted its peers(companies in the same industry)?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {CIK: '0000318154'})-[:IMPACTS]->(e:Event)-[:EVENT_HAS_TYPE]->(et:Event_Type),
      (c)-[:BELONGS_TO_INDUSTRY_OF]->(i:Industry)<-[:BELONGS_TO_INDUSTRY_OF]-(peers:Company)-[:IMPACTS]->(e)
RETURN c, i, e, et, peers


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: CIK)} {position: line: 1, column: 19, offset: 18} for query: "MATCH (c:Company {CIK: '0000318154'})-[:IMPACTS]->(e:Event)-[:EVENT_HAS_TYPE]->(et:Event_Type),\n      (c)-[:BELONGS_TO_INDUSTRY_OF]->(i:Industry)<-[:BELONGS_TO_INDUSTRY_OF]-(peers:Company)-[:IMPACTS]->(e)\nRETURN c, i, e, et, peers"


Full Context:
[]

> Finished chain.


{'query': 'About the company with CIK 0000318154, how have past events impacted the company and its sector? is the company exposed to specific political or regulatory risks that have historically impacted its peers(companies in the same industry)?',
 'result': "I don't know the answer.",
 'intermediate_steps': [{'query': "MATCH (c:Company {CIK: '0000318154'})-[:IMPACTS]->(e:Event)-[:EVENT_HAS_TYPE]->(et:Event_Type),\n      (c)-[:BELONGS_TO_INDUSTRY_OF]->(i:Industry)<-[:BELONGS_TO_INDUSTRY_OF]-(peers:Company)-[:IMPACTS]->(e)\nRETURN c, i, e, et, peers"},
  {'context': []}]}

In [24]:
chain.invoke({"query": "which companies did the event 2016_0 impacted?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (:Event {name: "2016_0"})-[:IMPACTS]->(c:Company)
RETURN c.name;
Full Context:
[{'c.name': '0000006201'}, {'c.name': '0000919012'}, {'c.name': '0000006951'}, {'c.name': '0000318154'}, {'c.name': '0001047127'}, {'c.name': '0001596532'}, {'c.name': '0001023024'}, {'c.name': '0000014707'}, {'c.name': '0001262039'}, {'c.name': '0000882095'}]

> Finished chain.


{'query': 'which companies did the event 2016_0 impacted?',
 'result': "I don't know the answer.",
 'intermediate_steps': [{'query': 'MATCH (:Event {name: "2016_0"})-[:IMPACTS]->(c:Company)\nRETURN c.name;'},
  {'context': [{'c.name': '0000006201'},
    {'c.name': '0000919012'},
    {'c.name': '0000006951'},
    {'c.name': '0000318154'},
    {'c.name': '0001047127'},
    {'c.name': '0001596532'},
    {'c.name': '0001023024'},
    {'c.name': '0000014707'},
    {'c.name': '0001262039'},
    {'c.name': '0000882095'}]}]}